# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
# Explore the data
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
#Summary
spaceship.describe()


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [6]:
#Check Number of Nulls
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
#your code here
# Handle missing values and feature engineering
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Drop columns that won't help with prediction
spaceship_clean = spaceship.drop(['PassengerId', 'Name', 'Cabin'], axis=1)

# Handle categorical variables - convert to string first to avoid mixed types
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

for col in categorical_cols:
    # Convert to string, fill missing values, then encode
    spaceship_clean[col] = spaceship_clean[col].astype(str).fillna('Unknown')
    le = LabelEncoder()
    spaceship_clean[col] = le.fit_transform(spaceship_clean[col])

# Handle numerical columns - fill with median
numerical_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in numerical_cols:
    spaceship_clean[col] = spaceship_clean[col].fillna(spaceship_clean[col].median())

# Separate features and target
X = spaceship_clean.drop('Transported', axis=1)
y = spaceship_clean['Transported'].astype(int)  # Convert boolean to int

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature names: {X.columns.tolist()}")

Features shape: (8693, 10)
Target shape: (8693,)
Feature names: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


**Perform Train Test Split**

In [9]:
#your code here
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target distribution in training: {np.bincount(y_train)}")
print(f"Target distribution in test: {np.bincount(y_test)}")

Training set size: (6954, 10)
Test set size: (1739, 10)
Target distribution in training: [3452 3502]
Target distribution in test: [863 876]


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [11]:
#your code here
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Bagging
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42
)

bagging_model.fit(X_train, y_train)
bagging_pred = bagging_model.predict(X_test)
bagging_accuracy = accuracy_score(y_test, bagging_pred)

print(f"Bagging Accuracy: {bagging_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, bagging_pred))

Bagging Accuracy: 0.7826

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       863
           1       0.79      0.78      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739



- Random Forests

In [12]:
#your code here
from sklearn.ensemble import RandomForestClassifier

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    min_samples_split=5
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print("\nFeature Importance:")
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance)

Random Forest Accuracy: 0.7987

Feature Importance:
        feature  importance
1     CryoSleep    0.189652
5   RoomService    0.147824
8           Spa    0.139881
9        VRDeck    0.139478
6     FoodCourt    0.114724
7  ShoppingMall    0.082926
0    HomePlanet    0.080678
3           Age    0.080104
2   Destination    0.018087
4           VIP    0.006646


- Gradient Boosting

In [13]:
#your code here
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)

print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")

Gradient Boosting Accuracy: 0.7993


- Adaptive Boosting

In [14]:
#your code here
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)
ada_accuracy = accuracy_score(y_test, ada_pred)

print(f"AdaBoost Accuracy: {ada_accuracy:.4f}")

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy: 0.7838


Which model is the best and why?

In [ ]:
#comment here
# Compare all models
results = {
    'Bagging': bagging_accuracy,
    'Random Forest': rf_accuracy,
    'Gradient Boosting': gb_accuracy,
    'AdaBoost': ada_accuracy
}

# Create comparison DataFrame
comparison_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
comparison_df = comparison_df.sort_values('Accuracy', ascending=False)

print("Model Comparison:")
print(comparison_df)

# Best model analysis
best_model = comparison_df.iloc[0]['Model']
best_accuracy = comparison_df.iloc[0]['Accuracy']

print(f"\nBest Model: {best_model} with accuracy: {best_accuracy:.4f}")

Model Comparison:
               Model  Accuracy
2  Gradient Boosting  0.799310
1      Random Forest  0.798735
3           AdaBoost  0.783784
0            Bagging  0.782634


In [ ]:
# Based on the results, Gradient Boosting performed best (79.93% accuracy), followed closely by Random Forest (79.87%)

# Gradient Boosting won because:
# 1. Sequential learning allows it to correct errors from previous models
# 2. Good at handling complex patterns in the data
# 3. Effective with the mix of categorical and numerical features
# 4. Less prone to overfitting with proper hyperparameters (max_depth=3, learning_rate=0.1)

# Random Forest was very close because:
# 1. Excellent at reducing overfitting through random feature selection
# 2. Handles mixed data types well (categorical + numerical)
# 3. Robust and less sensitive to hyperparameters
# 4. Good feature importance insights

# The close performance suggests both models captured the underlying patterns well
# Gradient Boosting's slight edge comes from its ability to iteratively improve predictions